# v5 One-Shot Kaggle Train

Use this notebook for the final v5 run. Do not use the older `kaggle_e4b_qlora.ipynb` proof-gate cells for v5; they contain legacy XML probe logic.

Kaggle setup: GPU T4 x2, Internet on, `HF_TOKEN` secret with write scope, Gemma license accepted. Run cells top to bottom. If Cell 2 installs packages and Kaggle asks for restart, restart and rerun from Cell 1.


In [ ]:
# Cell 1 - locked v5 config
import os
MODEL = 'gemma4_e4b'
ENGINE = 'unsloth'
HF_REPO = 'unsloth/gemma-4-E4B-it'
CHESS_REPO_URL = 'https://github.com/RyanDev1st/llm-and-engine.git'
CHESS_BRANCH = 'feat/v5-pure-chess'
WORKDIR = '/kaggle/working/llm-and-engine'
OUTPUT = 'gemma4_chess_e4b_kaggle'
CKPT_REPO = 'RyanDev1st/gemma4-chesscoach-ckpt-v5'
os.environ['CHESS_CKPT_REPO'] = CKPT_REPO
print('config', MODEL, ENGINE, CHESS_BRANCH, 'output', OUTPUT, 'ckpt', CKPT_REPO)


In [ ]:
# Cell 2 - GPU and deps
import subprocess, sys, shutil, torch
assert torch.cuda.is_available(), 'No GPU: set Kaggle Accelerator to GPU T4 x2, then rerun.'
print(subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total,memory.free', '--format=csv'], capture_output=True, text=True).stdout if shutil.which('nvidia-smi') else 'nvidia-smi missing')
print('cuda', torch.version.cuda, 'devices', torch.cuda.device_count())
assert torch.cuda.device_count() >= 2, 'This locked run expects T4 x2. If forced to 1 GPU, stop and ask before changing DDP settings.'
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--upgrade', 'unsloth', 'unsloth_zoo', 'bitsandbytes', 'accelerate', 'huggingface_hub', 'python-chess'], check=True)
print('deps installed; restart kernel if Kaggle shows a restart banner, then rerun from Cell 1')


In [ ]:
# Cell 3 - clone exact branch and set PYTHONPATH
import os, subprocess
def run(cmd, **kw):
    print('>', ' '.join(cmd)); return subprocess.run(cmd, check=True, text=True, **kw)
env = {**os.environ, 'GIT_LFS_SKIP_SMUDGE': '1'}
if not os.path.exists(WORKDIR):
    run(['git', 'clone', '--depth', '1', '--branch', CHESS_BRANCH, CHESS_REPO_URL, WORKDIR], env=env)
else:
    run(['git', '-C', WORKDIR, 'fetch', 'origin', CHESS_BRANCH, '--depth', '1'], env=env)
    run(['git', '-C', WORKDIR, 'checkout', '-B', CHESS_BRANCH, 'FETCH_HEAD'], env=env)
    run(['git', '-C', WORKDIR, 'reset', '--hard', 'FETCH_HEAD'], env=env)
run(['git', '-C', WORKDIR, 'log', '-1', '--oneline'])
os.environ['PYTHONPATH'] = f'{WORKDIR}/src/llm'
print('PYTHONPATH=', os.environ['PYTHONPATH'])


In [ ]:
# Cell 4 - HF login and E4B base download
import os
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login, snapshot_download
login(UserSecretsClient().get_secret('HF_TOKEN'))
dest = f'{WORKDIR}/src/llm/models/{MODEL}'
snapshot_download(repo_id=HF_REPO, local_dir=dest, allow_patterns=['*.json', '*.safetensors', '*.model', '*.txt', 'tokenizer*'])
print('base model at', dest)
print(sorted(os.listdir(dest))[:20])


In [ ]:
# Cell 5 - CPU preflight against the exact v5 split and E4B tokenizer
import subprocess, sys, os
env = {**os.environ, 'PYTHONPATH': f'{WORKDIR}/src/llm'}
subprocess.run([sys.executable, '-m', 'llm_training.kaggle_v5_one_shot', 'preflight', '--workdir', WORKDIR], check=True, cwd=WORKDIR, env=env)
print('PREFLIGHT OK: corpus + native render/mask/tokenizer gates passed')


In [ ]:
# Cell 6 - 20-step smoke train. Must pass before final. Never pushes to CKPT_REPO.
import subprocess, sys, os
env = {**os.environ, 'PYTHONPATH': f'{WORKDIR}/src/llm'}
subprocess.run([sys.executable, '-m', 'llm_training.kaggle_v5_one_shot', 'smoke', '--workdir', WORKDIR], check=True, cwd=WORKDIR, env=env)
print('SMOKE OK: now restart kernel if GPU memory is not fully free, then rerun Cells 1-5 and Cell 7')


In [ ]:
# Cell 7 - final train. First session: RESUME=False. Later sessions: set RESUME=True.
import subprocess, sys, os, torch
RESUME = False
free, total = torch.cuda.mem_get_info(0)
free_gib = free / 1024**3
print(f'GPU0 free: {free_gib:.1f} GiB / {total/1024**3:.1f} GiB')
assert free_gib >= 11.0, 'GPU memory is not clean. Restart kernel, rerun Cells 1-5, then rerun this cell.'
cmd = [sys.executable, '-m', 'llm_training.kaggle_v5_one_shot', 'train', '--workdir', WORKDIR]
if RESUME:
    cmd.append('--resume')
env = {**os.environ, 'PYTHONPATH': f'{WORKDIR}/src/llm'}
subprocess.run(cmd, check=True, cwd=WORKDIR, env=env)


In [ ]:
# Cell 8 - package adapter for local download after training or timeout
import os, shutil
run_dir = f'{WORKDIR}/runs/{OUTPUT}'
print('run dir contents:', os.listdir(run_dir) if os.path.isdir(run_dir) else 'MISSING')
out_zip = f'/kaggle/working/{OUTPUT}'
shutil.make_archive(out_zip, 'zip', run_dir)
size_mb = os.path.getsize(out_zip + '.zip') / 1e6
print(f'{out_zip}.zip ({size_mb:.1f} MB). Download from the Output panel.')
